In [1]:
import pickle
from typing import Literal
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root
import tqdm
import tqdm.auto
tqdm.auto.tqdm = tqdm.tqdm
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import collections

from python.config import load_config, SingleStepConfig, Config

cfg = load_config(os.environ.get('CONFIG', 'config.json'))

import python.plots as plots
from python.error_metrics import LoadedErrorMetrics, FilteredErrorMetrics, MetricsIndex as MI # Alias
from python.sampled_hierarchies import SampledHierarchies, SingleStepHierarchies, PartitionStepHierarchies
from python.statistical_test import slope_ttest, page_trend_test, CombineMethod, wilcoxon_test, TrendType, sign_test, per_series_test
from python.aggregation import Aggregation
from python.base_forecasting import MODEL_MIN_TRAIN_OBS
import python.utils as utils

from itertools import product

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

from python.time_series import TimeSeriesType, TimeSeries
from python.error_metrics import ErrorMetrics, MergedErrorMetrics, FilteredErrorMetrics
from python.aggregation import TOP_LEVEL, BOTTOM_LEVEL

In [2]:
if cfg.DATASET.NAME == 'web-traffic':
    X_raw = utils.fetch_dataset_web_traffic()
    X = utils.filter_last_days_available(X_raw, 28*100)
if cfg.DATASET.NAME == 'M5':
    X = utils.fetch_dataset_m5()
if cfg.DATASET.NAME == 'M4':
    X = utils.fetch_dataset_m4()
if cfg.DATASET.NAME == 'sim':
    X = utils.fetch_dataset_simulated()

X = utils.filter_last_days_available(X, cfg.HORIZON)
X = utils.reduce_dataset(X, start=cfg.DATASET.START, end=cfg.DATASET.END)
X = utils.align_fill_and_trim_series(X)
X = utils.filter_min_length(X, cfg.HORIZON + MODEL_MIN_TRAIN_OBS[cfg.MODEL] * cfg.LEVELS.BASE_PERIOD
)

In [3]:
AVAILABLE_LEVELS = Aggregation.complete_summing_matrix(n=cfg.LEVELS.SUBSET, seed=42)

if cfg.is_single_step_hierarchies():
    if cfg.LEVELS.PARTITION == 0:
        sh = SingleStepHierarchies(AVAILABLE_LEVELS)
    else:
        sh = PartitionStepHierarchies(AVAILABLE_LEVELS, cfg.LEVELS.PARTITION, cfg.LEVELS.BASE_PERIOD)
else:
    sh = SampledHierarchies(AVAILABLE_LEVELS, cfg.LEVELS.DIFF, cfg.LEVELS.SEQUENCES, max_h=cfg.LEVELS.MAX_H)

datasets = ['sim', 'web-traffic', 'M4', 'M5']
models = ['AutoETS', 'AutoARIMA', 'Prophet']

base_cfg = cfg.model_copy()
base_cfg.LEVELS = SingleStepConfig(base_period=cfg.LEVELS.BASE_PERIOD,
                                   single_step_repeats=0,)
base_cfg.model_post_init(None)

In [4]:
sh.full_hierarchy.resample_time_series(X)
for h in sh.hierarchies:
    h.copy_data(sh.full_hierarchy)
    for g in (TOP_LEVEL, BOTTOM_LEVEL):
        h[g][TimeSeriesType.TRAIN_SET] = h[g][TimeSeriesType.FULL_DATA].wide.head(-cfg.HORIZON).copy()

In [5]:
def agg_to_comp_hier(agg: str):
    for h, levels in base_sh.hierarchy_level_names.items():
        if agg in levels:
            for h_o in base_sh.hierarchies:
                if str(h_o) == h:
                    return h_o

In [6]:
with open(f"sample/{base_cfg.FOLDER}/sampled_hierarchies_0-500.pkl", 'rb') as f:
    base_sh = pickle.load(f)

In [7]:
base_sh.hierarchies[0][BOTTOM_LEVEL][TimeSeriesType.RECONCILED_FORECASTS].wide

,S0000,S0001,S0002,S0003,S0004,S0005,S0006,S0007,S0008,S0009,S0010,S0011,S0012,S0013,S0014,S0015,S0016,S0017,S0018,S0019,S0020,S0021,S0022,S0023,S0024,S0025,S0026,S0027,S0028,S0029,S0030,S0031,S0032,S0033,S0034,S0035,S0036,S0037,S0038,S0039,S0040,S0041,S0042,S0043,S0044,S0045,S0046,S0047,S0048,S0049,S0050,S0051,S0052,S0053,S0054,S0055,S0056,S0057,S0058,S0059,S0060,S0061,S0062,S0063,S0064,S0065,S0066,S0067,S0068,S0069,S0070,S0071,S0072,S0073,S0074,S0075,S0076,S0077,S0078,S0079,S0080,S0081,S0082,S0083,S0084,S0085,S0086,S0087,S0088,S0089,S0090,S0091,S0092,S0093,S0094,S0095,S0096,S0097,S0098,S0099,S0100,S0101,S0102,S0103,S0104,S0105,S0106,S0107,S0108,S0109,S0110,S0111,S0112,S0113,S0114,S0115,S0116,S0117,S0118,S0119,S0120,S0121,S0122,S0123,S0124,S0125,S0126,S0127,S0128,S0129,S0130,S0131,S0132,S0133,S0134,S0135,S0136,S0137,S0138,S0139,S0140,S0141,S0142,S0143,S0144,S0145,S0146,S0147,S0148,S0149,S0150,S0151,S0152,S0153,S0154,S0155,S0156,S0157,S0158,S0159,S0160,S0161,S0162,S0163,S0164,S0165,S0166,S0167,S0168,S0169,S0170,S0171,S0172,S0173,S0174,S0175,S0176,S0177,S0178,S0179,S0180,S0181,S0182,S0183,S0184,S0185,S0186,S0187,S0188,S0189,S0190,S0191,S0192,S0193,S0194,S0195,S0196,S0197,S0198,S0199,S0200,S0201,S0202,S0203,S0204,S0205,S0206,S0207,S0208,S0209,S0210,S0211,S0212,S0213,S0214,S0215,S0216,S0217,S0218,S0219,S0220,S0221,S0222,S0223,S0224,S0225,S0226,S0227,S0228,S0229,S0230,S0231,S0232,S0233,S0234,S0235,S0236,S0237,S0238,S0239,S0240,S0241,S0242,S0243,S0244,S0245,S0246,S0247,S0248,S0249,S0250,S0251,S0252,S0253,S0254,S0255,S0256,S0257,S0258,S0259,S0260,S0261,S0262,S0263,S0264,S0265,S0266,S0267,S0268,S0269,S0270,S0271,S0272,S0273,S0274,S0275,S0276,S0277,S0278,S0279,S0280,S0281,S0282,S0283,S0284,S0285,S0286,S0287,S0288,S0289,S0290,S0291,S0292,S0293,S0294,S0295,S0296,S0297,S0298,S0299,S0300,S0301,S0302,S0303,S0304,S0305,S0306,S0307,S0308,S0309,S0310,S0311,S0312,S0313,S0314,S0315,S0316,S0317,S0318,S0319,S0320,S0321,S0322,S0323,S0324,S0325,S0326,S0327,S0328,S0329,S0330,S0331,S0332,S0333,S0334,S0335,S0336,S0337,S0338,S0339,S0340,S0341,S0342,S0343,S0344,S0345,S0346,S0347,S0348,S0349,S0350,S0351,S0352,S0353,S0354,S0355,S0356,S0357,S0358,S0359,S0360,S0361,S0362,S0363,S0364,S0365,S0366,S0367,S0368,S0369,S0370,S0371,S0372,S0373,S0374,S0375,S0376,S0377,S0378,S0379,S0380,S0381,S0382,S0383,S0384,S0385,S0386,S0387,S0388,S0389,S0390,S0391,S0392,S0393,S0394,S0395,S0396,S0397,S0398,S0399,S0400,S0401,S0402,S0403,S0404,S0405,S0406,S0407,S0408,S0409,S0410,S0411,S0412,S0413,S0414,S0415,S0416,S0417,S0418,S0419,S0420,S0421,S0422,S0423,S0424,S0425,S0426,S0427,S0428,S0429,S0430,S0431,S0432,S0433,S0434,S0435,S0436,S0437,S0438,S0439,S0440,S0441,S0442,S0443,S0444,S0445,S0446,S0447,S0448,S0449,S0450,S0451,S0452,S0453,S0454,S0455,S0456,S0457,S0458,S0459,S0460,S0461,S0462,S0463,S0464,S0465,S0466,S0467,S0468,S0469,S0470,S0471,S0472,S0473,S0474,S0475,S0476,S0477,S0478,S0479,S0480,S0481,S0482,S0483,S0484,S0485,S0486,S0487,S0488,S0489,S0490,S0491,S0492,S0493,S0494,S0495,S0496,S0497,S0498,S0499
ds,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2023-12-25,2.777220,-3.053794,602.798828,629.045288,34248.660156,286.234344,12.516962,1.657251e+06,-2976.804199,127.182831,226.682617,1.463938,78.682770,364.474091,-159.166702,395.629669,-11.496640,-12.125005,1382506.875,-1828.388672,3.143027,32349.511719,-1737.904785,163.034485,6.363840,-0.662891,-2.769430,8780.674805,37.116676,1.118583,12885.625977,421.533905,-21.183130,-268.107452,46.352783,-216.540604,125.518929,-28.737946,93.658508,84.285172,-84.738411,1323.024292,8331.022461,-227.79464

In [8]:
merge_to_aggs = sh.hierarchy_level_names

for merge_h in sh.hierarchies:
    comp_hs = [agg_to_comp_hier(a) for a in merge_to_aggs[str(merge_h)][1:-1]]
    for g in (TOP_LEVEL, BOTTOM_LEVEL):
        mean_df = pd.DataFrame(
            np.mean([h[g][TimeSeriesType.RECONCILED_FORECASTS].wide for h in comp_hs], axis=0),
            index=comp_hs[0][g][TimeSeriesType.RECONCILED_FORECASTS].wide.index,
            columns=comp_hs[0][g][TimeSeriesType.RECONCILED_FORECASTS].wide.columns,
        )
        merge_h[g]._time_series_library[TimeSeriesType.RECONCILED_FORECASTS] = TimeSeries(mean_df, g, method='avg')
        for s in (TimeSeriesType.BASE_FORECASTS, TimeSeriesType.VALIDATION_SET):
            merge_h[g]._time_series_library[s] = comp_hs[0][g][s]

In [9]:
merge_h[g]._time_series_library

{<TimeSeriesType.FULL_DATA: 'full_data'>: <python.time_series.TimeSeries at 0x78aa379e86e0>,
 <TimeSeriesType.TRAIN_SET: 'train_set'>: <python.time_series.TimeSeries at 0x78aa37a06a50>,
 <TimeSeriesType.RECONCILED_FORECASTS: 'reconciled_forecasts'>: <python.time_series.TimeSeries at 0x78aa379be1e0>,
 <TimeSeriesType.BASE_FORECASTS: 'base_forecasts'>: <python.time_series.TimeSeries at 0x78aa36e29460>,
 <TimeSeriesType.VALIDATION_SET: 'validation_set'>: <python.time_series.TimeSeries at 0x78aa36e29160>}

In [10]:
em = FilteredErrorMetrics(MergedErrorMetrics(
    [ErrorMetrics(h) for h in sh.hierarchies]
), MI(error_metric='nrmse', forecast_type='reconciled_forecasts')
)


In [11]:
em.save(f'metrics/{cfg.FOLDER}/avg_hiers_erros.csv')